# CrisisQ — SDCNL evaluation with frozen models

**Same Kaggle model inputs. Dataset downloaded automatically from its original source. No retraining or threshold changes.**

Internet ON; GPU not required. Attach the **same complete original model-containing notebook output** used for the runtime matrix: `CrisisQ_nonclinical_claim_verification`, containing `crisisq_nonclinical_v1_compat/models/saqs__42/{fp32.onnx,q_quint8.onnx}`. Leave other original model outputs attached if you used them before. Reports-only outputs do not contain weights. No input ZIP or manual dataset upload is needed.

The notebook fetches the authors' SDCNL CSVs from a fixed GitHub commit and verifies their checksums. It evaluates raw **title + two newlines + selftext**. Usernames, URLs and the source's cleaned-author features are excluded from model input.

**Primary:** authors'379-post test split after fixed exclusions. **Secondary:** all1,895 posts and raw-row sensitivity analyses. The authors' training split is never used to train or select CrisisQ here. Panels share messages and are not independent replications.

Requires the frozen primary FP32/QUInt8 pair; automatically includes every complete jointly accepted CE/SAQS seed pair actually available, with each original selected threshold/quantizer. That is2 passes if only the primary pair is available, up to28 model passes if all seven joint pairs are available. No runtime grid or threshold search.

SDCNL labels identify r/SuicideWatch versus r/Depression membership. Results measure agreement with those labels; depression-post alerts are not adjudicated false positives. This is a source-label evaluation, not clinical validation. Missing historical training texts prevent a guarantee of independence.


## 1. Paths

Keep defaults for the first run. Use new output **and temporary** folders after changing the host, input coverage or code. Settings are frozen before inference. The notebook uses ORT1.31.1, ALL graph optimization, four CPU threads, the existing tokenizer and128-token limit.


In [ ]:
from pathlib import Path
import sys,os,json,subprocess,importlib.util
assert sys.version_info >= (3,11), 'Use Python3.11+ with matching pinned wheels.'
assert Path('/kaggle/input').exists(), 'Run in Kaggle with the original whole model output attached.'
CONFIG={'input_root':'/kaggle/input','output_root':'/kaggle/working/crisisq_sdcnl_frozen_v1','temporary_root':'/kaggle/temp/crisisq_sdcnl_frozen_v1'}
OUTPUT=Path(CONFIG['output_root']);OUTPUT.mkdir(parents=True,exist_ok=True)
TEMP=Path(CONFIG['temporary_root']);TEMP.mkdir(parents=True,exist_ok=True)
BUNDLE=Path('/kaggle/working/crisisq_sdcnl_code_v1');BUNDLE.mkdir(parents=True,exist_ok=True)
print('Output:',OUTPUT)


## 2. Unpack verified helper code and frozen metadata

Embedded data includes the prior selection, frozen rules/tokenizer and previous evaluation hashes; **no SDCNL posts or neural weights are embedded**. The CSVs are downloaded from their original source in the next step.


In [ ]:
import base64,hashlib,io,zipfile
PAYLOAD='...'
raw=base64.b64decode(PAYLOAD)
assert hashlib.sha256(raw).hexdigest()=='5896c79eae4b2681ee6aea1c72fade8d8ef952dcfd0df1d1718e6ddce65c2ead'
allowed=['HOW_TO_RUN.md', 'analyze.py', 'environment.py', 'external_frozen.json', 'frozen_rules.json', 'full_evaluation_seed.json.gz', 'selection_frozen.json', 'source_manifest.json', 'study.py', 'worker.py']
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert set(z.namelist())==set(allowed)
    for name in allowed:
        assert Path(name).name==name
        (BUNDLE/name).write_bytes(z.read(name))
del raw,PAYLOAD
sys.path.insert(0,str(BUNDLE))
def helper(name):
    spec=importlib.util.spec_from_file_location('sdcnl_'+name,BUNDLE/(name+'.py'))
    module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module);return module
print('Verified',len(allowed),'embedded helper/data files.')


## 3. Download, check overlap, tokenize and freeze

The isolated preparation environment uses NumPy2.1.3 and tokenizers0.23.3. It is created **without ensurepip**, using the previously fixed Kaggle-compatible approach. The notebook kernel's packages stay unchanged.

Source commit: `ddf995aabd028657385cac6cee92c5d53774992d`. Downloads all three original CSVs and verifies combined=train∪test. Primary eligibility excludes known exact title/body overlaps, conflicting labels, empty/removed bodies and duplicate normalized full texts. Duplicate representatives prefer the authors' test split; no results are used for exclusions.

Overlap checks inspect recognized text/text_hash fields in mounted CSV/JSONL files up to256MiB; unsupported/larger files are listed. This is **exact overlap detection only**, with incomplete historical training coverage. Missing original training texts do not block evaluation, but prevent a clean independence claim. No near-duplicate exclusion is claimed.

The frozen gate reads the full post; the neural model retains its existing first128-token input. Truncation is recorded instead of changing the model or input limit.


In [ ]:
envtools=helper('environment');PREP=envtools.ensure_environment(TEMP/'preparation_environment')
probe=subprocess.run([str(PREP),'-I','-c','import numpy,tokenizers;assert numpy.__version__=="2.1.3" and tokenizers.__version__=="0.23.3"'],capture_output=True,text=True)
if probe.returncode:
    log=OUTPUT/'preparation_install.log'
    with log.open('w') as f:
        status=subprocess.run(envtools.pip_command(PREP,'install','--only-binary=:all:','numpy==2.1.3','tokenizers==0.23.3'),stdout=f,stderr=subprocess.STDOUT)
    if status.returncode:raise RuntimeError(log.read_text()[-3000:])
CFG=TEMP/'configuration.json';CFG.write_text(json.dumps(CONFIG,indent=2))
subprocess.run([str(PREP),'-I',str(BUNDLE/'study.py'),str(CFG)],check=True)
study=helper('study');analyzer=helper('analyze')
frozen=json.loads((OUTPUT/'protocol_frozen.json').read_text());plan=frozen['protocol']
print('Plan frozen before inference:',frozen['plan_hash'])
print('Original source:',plan['source']['repository'],'commit:',plan['source']['commit'])
print('Available models:',len(plan['models']))
for row in plan['coverage']:print(row)
print('Thresholds — copied unchanged from prior selection:')
for m in plan['models']:print(m['id'],m['tau'])


## 4. Install the fixed isolated inference runtime

ORT1.31.1 and NumPy2.1.3. Native hashes/build information and installed versions are saved. A missing pinned wheel stops the run instead of changing versions. No model is downloaded.


In [ ]:
def incomplete(phase,error):
    (OUTPUT/'failure.json').write_text(json.dumps(dict(complete=False,phase=phase,error=str(error)),indent=2))
    print('INCOMPLETE diagnostics:',analyzer.pack(OUTPUT))
try:RUNTIME=study.install(CONFIG)
except Exception as error:
    incomplete('installation',error);raise
print('Ready. No dataset scoring has run yet.')


## 5. Score all available frozen graphs once

Every message is scored once per graph. Neural and hybrid decisions reuse the same probabilities; gate exits do not require a second neural pass. One-case inference preserves the existing numerical protocol. Jobs checkpoint every64 messages and verify graph/input/runtime hashes. A fixed16-case subset checks within-session repeatability.

Do not select a favorable seed or runtime after seeing SDCNL scores. The additional CE/SAQS pairs are conditional on joint acceptance in the original development study, which can bias arm comparisons. Their original selected quantizers and thresholds may differ; this is a comparison of frozen selected systems.

Resume only on the same unchanged host/session. Preserve the report if the session will expire.


In [ ]:
try:study.execute(CONFIG,RUNTIME)
except Exception as error:
    incomplete('inference',error);raise
print('All frozen graph passes completed.')


## 6. Analyze and download

Reports include source-positive recall, alerts on source-negative posts, source-label precision/F1/agreement, gate effects, and positives lost/gained after quantization. Message-sampling intervals resample author clusters1,000 times. Separate paired-seed intervals summarize CE–SAQS across jointly accepted available pairs; seeds are never pooled as independent posts.

Read the primary test panel first. All-post/raw panels are sensitivity analyses. Check `dataset_summary.json` for exclusions/truncation and `overlap_coverage.json` before describing the result as external. Reports contain no raw posts, titles, author strings or source URLs.


In [ ]:
try:COMPLETION=analyzer.main(OUTPUT)
except Exception as error:
    incomplete('analysis',error);raise
assert COMPLETION['all_jobs_complete']
print(json.dumps(json.loads((OUTPUT/'dataset_summary.json').read_text()),indent=2))
import csv
with (OUTPUT/'metrics.csv').open() as f:
    for r in csv.DictReader(f):
        if r['panel']=='authors_test_clean_primary' and r['model'].startswith('saqs__42__'):
            print(r['model'],r['mode'],'N=',r['n'],'source-positive recall=',r['source_positive_recall'],'source-negative alert rate=',r['source_negative_alert_rate'])
print((OUTPUT/'READ_ME_results.txt').read_text())
REPORT=OUTPUT.parent/(OUTPUT.name+'_reports.zip')
with zipfile.ZipFile(REPORT) as z:
    assert z.testzip() is None
    assert not any(n.endswith('.onnx') or n.endswith('set.csv') for n in z.namelist())
from IPython.display import display,FileLink
print('RETURN THIS FILE:',REPORT)
display(FileLink(str(REPORT)))


Original sources: [SDCNL authors' repository](https://github.com/ayaanzhaque/SDCNL), [original CSV directory](https://github.com/ayaanzhaque/SDCNL/tree/main/data), [paper](https://arxiv.org/abs/2102.09427). The repository describes subreddit-derived labels. Check the source's applicable data-use terms before redistributing original posts; this notebook keeps them out of reports.
